In [1]:
import sys

sys.path.append(r"C:\Users\david\PycharmProjects\normalizing-flows")
sys.path.append(r"C:\Users\david\PycharmProjects\nfmc")
sys.path.append(r"C:\Users\david\PycharmProjects\mcmc-diagnostics")
sys.path.append(r"C:\Users\david\PycharmProjects\potentials")
sys.path.append(r"C:\Users\david\PycharmProjects\potentials")

In [2]:
import torch
from nfmc.util import sum_except_batch


class DiagonalGaussian:
    """
    Class for the diagonal Gaussian distribution.
    """

    def __init__(self, event_shape, mu: float = 3.0, std: float = 2.0):
        self.event_shape = event_shape
        self.mu = mu
        self.std = std

    @property
    def first_moment(self):
        return torch.full(size=self.event_shape, fill_value=self.mu)

    @property
    def variance(self):
        return torch.full(size=self.event_shape, fill_value=self.std**2)

    @property
    def second_moment(self):
        return self.variance + self.first_moment**2

    def neg_log_prob(self, x: torch.Tensor):
        """
        Computes the negative log probability density of this distribution.

        :param torch.Tensor x: input tensor with shape `(*batch_shape, *event_shape)`.
        :return: negative log probability density tensor with shape `batch_shape`.
        """
        return sum_except_batch(
            (x - self.mu) ** 2 / (2 * self.std**2), self.event_shape
        )


In [12]:
torch.manual_seed(0)

event_shape = (4,)
n_chains = 4
target = DiagonalGaussian(event_shape, mu=0.5, std=0.5)

In [13]:
from nfmc.algorithms.mh.base import MHSampler
from nfmc.algorithms.mh.imh import IMHKernel
from nfmc.algorithms.preconditioning.base import PreconditionedMCMCSampler
from nfmc.algorithms.preconditioning.preconditioners import NormalizingFlowPreconditioner
from torchflows import Flow, RealNVP

torch.manual_seed(0)

imh_flow = Flow(RealNVP(event_shape))
# imh_preconditioner = NormalizingFlowPreconditioner(imh_flow)
imh_kernel = IMHKernel(event_shape=imh_flow.event_shape, neg_log_prob_target=target.neg_log_prob)
# imh_sampler = PreconditionedMCMCSampler(imh_kernel, imh_preconditioner)
imh_sampler = MHSampler(imh_kernel)

z0 = torch.rand(size=(n_chains, *event_shape)) * 2 - 1
# imh_warmup_draws, imh_latent_warmup_draws = imh_sampler.warmup(
#     z0=z0,
#     n_steps=1200,
#     preconditioner_update_interval=500,
#     return_latent_samples=True,
# )
imh_sampling_draws = imh_sampler.sample(z0, n_steps=20000)

Sampling: 100%|██████████| 20000/20000 [00:30<00:00, 648.58it/s, IMH, 5188.583 c/s, 0.000 g/s, 0.124 acc]


In [14]:
import matplotlib.pyplot as plt

torch.manual_seed(0)

x_imh_flow = imh_flow.sample((10000)).detach()
x_imh_flow.mean(0)

tensor([ 0.4588, -2.4516,  5.0560, -2.7342])

In [15]:
imh_sampling_draws.as_tensor()[100:200].mean(dim=(0, 1))

tensor([0.4699, 0.5436, 0.3671, 0.6084])

In [16]:
imh_sampling_draws.as_tensor().mean(dim=(0, 1))

tensor([0.5032, 0.5028, 0.4984, 0.5035])

In [19]:
imh_sampling_draws.as_tensor().var(dim=(0, 1)).sqrt()

tensor([0.4941, 0.4952, 0.5019, 0.4964])

In [28]:
imh_latent_warmup_draws.as_tensor()[100:200].mean(dim=(0, 1))

tensor([1.0731, 1.8980, 0.4044, 0.4284])

In [20]:
from nfmc.algorithms.preconditioning.samplers.neutra import NeuTraHMC


torch.manual_seed(0)

flow = Flow(RealNVP(event_shape))
sampler = NeuTraHMC(flow, target.neg_log_prob)

z0 = torch.rand(size=(n_chains, *event_shape)) * 2 - 1
warmup_draws, latent_warmup_draws = sampler.warmup(
    z0=z0,
    n_steps=1200,
    preconditioner_update_interval=500,
    return_latent_samples=True,
)
sampling_draws = sampler.sample(z0=latent_warmup_draws.last_sample, n_steps=2000)

Sampling: 100%|██████████| 2000/2000 [03:40<00:00,  9.07it/s, HMC, 1523.556 c/s, 1451.006 g/s, 0.618 acc]


In [25]:
warmup_draws.as_tensor()[100:200].mean(dim=(0, 1))

tensor([3.1382, 3.0180, 3.0092, 3.0487])

In [24]:
sampling_draws.as_tensor().mean(dim=(0, 1))

tensor([3.0407, 3.0172, 3.0649, 2.9974])